# Load MEXT Education Content Data

This notebook loads the **Children's Learning Support Site Content Information** (子供の学び応援サイト掲載コンテンツ情報) CSV into a Delta table.

- Source: [e-Gov Data Portal](https://data.e-gov.go.jp/data/en/dataset/mext_20210222_0025)
- Publisher: Ministry of Education, Culture, Sports, Science and Technology (MEXT / 文部科学省)
- License: CC BY 4.0
- Encoding: Shift-JIS → converted to UTF-8 during read

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Define schema matching the MEXT CSV columns
schema = StructType([
    StructField("教材_ID", StringType(), True),
    StructField("教材_名称", StringType(), True),
    StructField("教材_言語", StringType(), True),
    StructField("教材_キーワード", StringType(), True),
    StructField("教材_形式", StringType(), True),
    StructField("教材_対象者", StringType(), True),
    StructField("教材_教科等", StringType(), True),
    StructField("教材_分野_科目", StringType(), True),
    StructField("教材_対象学年", StringType(), True),
    StructField("教材_コンテンツ形式", StringType(), True),
    StructField("教材_URL", StringType(), True),
    StructField("教材_価格_区分", StringType(), True),
    StructField("教材_ライセンス", StringType(), True),
    StructField("教材_状態", StringType(), True),
    StructField("教材_公開者", StringType(), True)
])

print('Schema defined with', len(schema.fields), 'columns')

In [ ]:
# Create schema and load CSV into Delta table
spark.sql('CREATE SCHEMA IF NOT EXISTS mext')
print('Schema mext ready.')

csv_path = 'Files/mext/mext_education_content.csv'

df = (spark.read
    .format('csv')
    .option('header', 'true')
    .option('encoding', 'shift_jis')
    .schema(schema)
    .load(csv_path))

# Drop rows where ID is null (empty padding rows)
df = df.filter(col('教材_ID').isNotNull())

print(f'Loaded {df.count()} rows from CSV')
df.printSchema()
df.show(5, truncate=30)

In [ ]:
# Write to Delta table
df.write.mode('overwrite').format('delta').saveAsTable('mext.education_content')

print('Data written to mext.education_content')

# Verify row count
count = spark.sql('SELECT COUNT(*) as cnt FROM mext.education_content').collect()[0]['cnt']
print(f'Verified: {count} rows in mext.education_content')

# Show subject breakdown
print('\nRows by subject (教科):')
spark.sql("""
    SELECT 教材_教科等 AS subject, COUNT(*) AS count
    FROM mext.education_content
    GROUP BY 教材_教科等
    ORDER BY count DESC
""").show(truncate=False)